# T8 · Train your own tiny transformer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-transformer/notebooks/T8-train-your-own.ipynb)

In this notebook you train **our tiny transformer** from scratch: the same 9,417-number model, the same Shakespeare text and the
same recipe that made the model you met in the course. You watch its loss fall, measure how often it guesses the next character
right, make it write, and save the trained model to an `outputs/` folder.

**How to run it**
1. Open it in Google Colab (the badge above) and sign in with your Google account when Colab asks.
2. Choose **Runtime → Run all**, or press ▶ on each cell from top to bottom.
3. No GPU is needed: the model is tiny, so a CPU trains it quickly.

| | Quick run (the default) | Full run |
|---|---|---|
| Steps | 5,000 | 100,000, the course's run |
| Time on a CPU | about 1 minute | about 16–20 minutes |
| Held-back loss | about 2.02 | about 1.93 |

Times were measured on the course's server; Colab's free CPU is similar, but times vary.

**Sections:** 1 set up · 2 get the data · 3 the model's code · 4 characters become numbers · 5 training examples ·
6 an untrained model · 7 the training recipe · 8 train · 9 watch it learn · 10 evaluate · 11 make it write · 12 save your model ·
13 try next.

Your run is your own experiment. On a CPU, its first 5,000 steps land on the course's recorded numbers; a GPU or another PyTorch
version gives slightly different numbers, which is normal.

**On your own computer:** `git clone https://github.com/gkiri/tiny_repo`, then `cd tiny_repo/tiny-transformer/notebooks`,
`pip install -r ../requirements.txt jupyter`, and open this notebook in Jupyter.

## 1 · Set up

The notebook needs three libraries: PyTorch, NumPy and Matplotlib. Colab already has them, so the install line finishes in seconds.
If pip adds that you may need to restart the kernel, you do not: nothing new was installed. Every file the notebook reads comes from the course's public labs repository on GitHub, or from your copy of it if you cloned it.

In [ ]:
import sys
from pathlib import Path

REPO = "https://raw.githubusercontent.com/gkiri/tiny_repo/main/"   # the course's public labs repository
CLONE = Path("../..")   # in a clone, this notebook sits in tiny_repo/tiny-transformer/notebooks


def source(path):
    """Where to read a repository file: your clone's copy if you have one, otherwise its address on GitHub."""
    local = CLONE / path
    return str(local) if local.exists() else REPO + path


REQUIREMENTS = source("tiny-transformer/requirements.txt")
print("installing from", REQUIREMENTS)

In [ ]:
%pip install -q -r {REQUIREMENTS}

In [ ]:
import hashlib, json, math, shutil, time, urllib.request

import matplotlib.pyplot as plt
import torch

torch.set_num_threads(1)   # the model is so small that one CPU thread is as fast as several
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} · Python {sys.version.split()[0]} · computing on: {device}")

## 2 · Get the data

**Tiny Shakespeare** is 1,115,394 characters of Shakespeare's plays, collected by Andrej Karpathy for char-rnn. The plays are in
the public domain. The cell downloads the course's copy and checks its fingerprint (sha256), so you train on exactly the text the
course used. If GitHub cannot be reached, it downloads the original instead, with the same check.

In [ ]:
SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"   # the fingerprint of the course's copy
ORIGINAL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"


def read(where):
    """The bytes of a local file or a web address."""
    if where.startswith("http"):
        with urllib.request.urlopen(where, timeout=60) as response:
            return response.read()
    return Path(where).read_bytes()


try:
    raw = read(source("tiny-transformer/data/tinyshakespeare.txt"))
except OSError:                                   # GitHub unreachable: use the original source
    raw = read(ORIGINAL)
assert hashlib.sha256(raw).hexdigest() == SHA256, "this is not the course's Tiny Shakespeare"
text = raw.decode("utf-8")
print(f"{len(text):,} characters · sha256 {SHA256[:12]}… ✓\n")
print(text[:250])

## 3 · The model's code

Everything the model is, in one file: the course's `model.py`, unchanged. Start with the diagram at the top, then read the parts
in order:

- `Config`: the model's sizes.
- `Tokenizer`: characters ↔ ids.
- `CausalSelfAttention`: each position looks back at earlier characters.
- `Block`: attention, then a feed-forward network. Each result is added to its row.
- `Prompter`: the whole model, meaning the two tables, one block and the readout. The class keeps its course file name; it is our
  tiny transformer.
- `generate`: write one character at a time.

Running the cell only defines these parts; nothing is computed yet.

In [ ]:
"""Our tiny transformer: 9,417 learned numbers that predict the next character of Shakespeare.

It reads up to 32 characters of text and gives a probability to each of the 65 characters that could come next. Writing is
that prediction in a loop: pick a character, append it, slide the 32-character window, and predict again. (The class is
called Prompter for the course's file names; learners meet it as "our tiny transformer".)

The architecture is the "10k" configuration of maxpolaczuk/tiny-character-transformer (a 1-block, 2-head character GPT on Tiny
Shakespeare). The code here is written fresh for the course (that repository has no licence, so none of its code or weights
are copied): every line is one the course teaches.

    characters ─► token table (65 × 28) + position table (32 × 28)        lookup:      what, and where
               ─► one block: LayerNorm ─► 2-head causal attention ─► +    look back:   which earlier letters matter
                             LayerNorm ─► feed-forward 28 → 56 → 28 ─► +  think:       transform each row on its own
               ─► final LayerNorm ─► readout with the token table again   score all 65 characters
               ─► softmax                                                  probabilities for the next character

Parameters: token table 1,820 · positions 896 · block 6,580 (two LayerNorms 112, q|k|v 2,436, attention out 812,
feed-forward 1,624 + 1,596) · final LayerNorm 56 · readout bias 65 = 9,417.
"""

import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    vocab_size: int = 65    # every distinct character in Tiny Shakespeare
    block_size: int = 32    # how many characters the model can read at once (its context)
    d_model: int = 28       # how many numbers describe one character in one place (a row)
    n_head: int = 2         # attention heads, each looking with 28 / 2 = 14 numbers
    d_ff: int = 56          # the feed-forward network's hidden width (2 × d_model)
    n_layer: int = 1        # transformer blocks

    def to_dict(self):
        return asdict(self)


class Tokenizer:
    """One id per character, in sorted order: '\\n' is 0, ' ' is 1, … 'z' is 64."""

    def __init__(self, text: str):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}

    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]

    def decode(self, ids) -> str:
        return "".join(self.chars[int(i)] for i in ids)


class CausalSelfAttention(nn.Module):
    """Each row asks a question (q), every earlier row offers a label (k) and a message (v). A row reads the messages of the
    rows whose labels match its question best, never a row to its right."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)     # q, k and v for both heads in one matrix
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        # mix the heads' results back into one row
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False)

    def forward(self, x, keep=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)          # how well each question matches each label
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))   # no peeking at the future
        att = F.softmax(att, dim=-1)                                          # shares that add up to 1
        if keep is not None:
            keep["attn"] = att.detach()
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)            # each row: a weighted mix of messages
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.d_model))

    def forward(self, x, keep=None):
        x = x + self.attn(self.ln1(x), keep)    # look back, and add what was found to the row
        x = x + self.ff(self.ln2(x))            # think about each row on its own, and add that too
        return x


class Prompter(nn.Module):
    def __init__(self, cfg: Config = Config()):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)    # one learned row per character
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)    # one learned row per place 0 … 31
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))  # the readout reuses tok_emb (tied); only its bias is new
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # small random numbers (σ = 0.02), so the untrained model's guesses are nearly even: its first loss is about
        # ln 65 = 4.17, the loss of a uniform guess over 65 characters
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, keep=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))   # what + where
        for b in self.blocks:
            x = b(x, keep)
        x = self.ln_f(x)
        logits = x @ self.tok_emb.weight.T + self.head_bias                         # a score for every character
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n, temperature=1.0, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

## 4 · Characters become numbers

The model reads numbers, not letters. The tokenizer gives each of the text's 65 distinct characters an id, in sorted order: `'\n'`
is 0, `' '` is 1, … `'z'` is 64. The first 90% of the text is for training. The last 10% is **held back** (validation): the model
never trains on it, so it shows how well the model does on text it has not seen.

In [ ]:
tok = Tokenizer(text)
ids = torch.tensor(tok.encode(text), dtype=torch.long)   # the whole text as ids
n_train = int(0.9 * len(ids))
train_ids, val_ids = ids[:n_train], ids[n_train:]       # first 90% trains · last 10% is held back

print(len(tok.chars), "characters:", repr("".join(tok.chars)))
print('"First Citizen" →', tok.encode("First Citizen"))
print(f"training: {len(train_ids):,} characters · held back: {len(val_ids):,} characters")

## 5 · Training examples

One example is a window of 32 characters (the input) and the same window shifted one place to the right (the targets). At every
position the model must predict the next character, so a window holds 32 questions. A batch stacks 64 random windows: 2,048
questions per training step.

In [ ]:
cfg = Config()          # vocabulary 65 · context 32 · width 28 · 2 heads · feed-forward 56 · 1 block
T = cfg.block_size      # characters per window: the model's context
B = 64                  # windows per batch (the recipe's batch_size)


def windows(data, starts):
    """Inputs x and targets y for the windows that begin at `starts`: y is x shifted one character to the right."""
    x = torch.stack([data[i:i + T] for i in starts])
    y = torch.stack([data[i + 1:i + T + 1] for i in starts])
    return x.to(device), y.to(device)


def random_starts(data, n, generator=None):
    """n random window starts. The bound len − T − 1 is the recorded run's: it skips the very last valid start (one of about
    a million), and keeping it lets your run follow the course's exactly."""
    return torch.randint(len(data) - T - 1, (n,), generator=generator)


demo = torch.Generator().manual_seed(0)   # a separate random stream, so this demo changes nothing later
x, y = windows(train_ids, random_starts(train_ids, B, demo))
print("a batch:", tuple(x.shape), "inputs and", tuple(y.shape), "targets\n")
print("input  :", repr(tok.decode(x[0])))
print("target :", repr(tok.decode(y[0])))

## 6 · An untrained model

The model starts with small random numbers, so it guesses almost evenly among the 65 characters. Its loss is then close to
ln 65 ≈ 4.17, the loss of an even guess, and its writing is gibberish. Here are its 9,417 learned numbers, table by table.

In [ ]:
torch.manual_seed(1337)
model = Prompter(cfg).to(device)
print(f"{count_params(model):,} learned numbers")
for name, p in sorted(model.named_parameters(), key=lambda kv: kv[0] == "head_bias"):   # the readout's bias last
    print(f"  {name:26s} {str(tuple(p.shape)):>9s} {p.numel():>6,}")

In [ ]:
x, y = windows(val_ids, random_starts(val_ids, B, torch.Generator().manual_seed(0)))
with torch.no_grad():
    _, loss = model(x, y)
print(f"untrained loss {loss.item():.3f} · even guess ln 65 = {math.log(cfg.vocab_size):.3f}\n")

start = torch.tensor([tok.encode("ROMEO:\n")], device=device)
print(tok.decode(model.generate(start, 120, generator=torch.Generator(device=device).manual_seed(42))[0]))

## 7 · The training recipe

This is the course's recorded recipe, unchanged:

- **Optimizer:** AdamW.
- **Learning rate:** it warms up from 0 to 0.003 over the first 200 steps, then follows a cosine curve down to 0.0003 at step 100,000.
- **Each step:** one batch of 64 random windows.
- **Every 500 steps, a check:** loss and accuracy on 100 fixed batches of training text and 100 of held-back text. The windows are
  the same every time, so two checks differ only because the model changed.
- **The kept model** is the one with the lowest held-back loss. Training stops early after 20 checks in a row without improvement.

`QUICK = True` runs the first 5,000 steps of exactly this recipe. Set `QUICK = False` for all 100,000 steps.

In [ ]:
QUICK = True    # True: the first 5,000 steps (about a minute) · False: the course's full 100,000 steps

RECIPE = dict(
    lr=3e-3,            # the highest learning rate, reached after the warm-up
    min_lr=3e-4,        # where the cosine curve ends
    warmup=200,         # steps that ramp the learning rate up from 0
    max_steps=100_000,  # the schedule's length: the course's full run
    batch_size=64,      # windows per step
    eval_every=500,     # steps between checks
    eval_batches=100,   # fixed batches per check (100 × 64 windows)
    patience=20,        # stop after this many checks without improvement
    seed=1337,          # the course's random seed
    eval_seed=2024,     # the seed of the fixed check batches
)
STEPS = 5_000 if QUICK else RECIPE["max_steps"]


def lr_at(step):
    """The learning rate at a step: a linear warm-up, then a cosine curve from lr down to min_lr."""
    r = RECIPE
    if step < r["warmup"]:
        return r["lr"] * step / r["warmup"]
    progress = min(1.0, (step - r["warmup"]) / max(1, r["max_steps"] - r["warmup"]))
    return r["min_lr"] + 0.5 * (r["lr"] - r["min_lr"]) * (1 + math.cos(math.pi * progress))

In [ ]:
check_stream = torch.Generator().manual_seed(RECIPE["eval_seed"])
SPLITS = {"train": train_ids, "val": val_ids}
FIXED = {split: [random_starts(data, B, check_stream) for _ in range(RECIPE["eval_batches"])]
         for split, data in SPLITS.items()}


@torch.no_grad()
def evaluate(model):
    """Average loss and accuracy on the fixed check batches. Accuracy: how often the most probable next character is right."""
    model.eval()
    result = {}
    for split, data in SPLITS.items():
        losses, hits = [], []
        for starts in FIXED[split]:
            x, y = windows(data, starts)
            logits, loss = model(x, y)
            losses.append(loss.item())
            hits.append((logits.argmax(-1) == y).float().mean().item())
        result[split] = sum(losses) / len(losses)
        result[split + "_acc"] = sum(hits) / len(hits)
    model.train()
    return result


print(f"{STEPS:,} steps · checks on {RECIPE['eval_batches']} × {B} fixed windows per split")
print("learning rate at steps 0, 200, 5,000 and 100,000:", ", ".join(f"{lr_at(s):.5f}" for s in (0, 200, 5_000, 100_000)))

## 8 · Train

Each step does four things:

1. Take a batch of windows.
2. Compute the loss: how surprised the model is by the right next characters.
3. Backpropagate, which gives every one of the 9,417 numbers a gradient.
4. Let AdamW nudge every number against its gradient.

Every 500 steps there is a check. A ★ marks a new best held-back loss, and the model's numbers at that point are the ones kept.

In [ ]:
torch.manual_seed(RECIPE["seed"])        # the course's seed: the same starting numbers, then the same batches
model = Prompter(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=RECIPE["lr"])

log, best, best_step, best_state, bad = [], float("inf"), 0, None, 0
t0 = time.time()
for step in range(STEPS + 1):
    if step % RECIPE["eval_every"] == 0:                         # a check on the fixed batches
        e = evaluate(model)
        better = e["val"] < best - 1e-4
        if better:                                               # a new best: keep a copy of its numbers
            best, best_step, bad = e["val"], step, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        elif step:
            bad += 1
        log.append({"step": step, "lr": lr_at(step), **{k: round(v, 5) for k, v in e.items()},
                    "seconds": round(time.time() - t0, 1)})
        print(f"step {step:7,} · train {e['train']:.4f} · held back {e['val']:.4f} · accuracy {e['val_acc']:.1%}"
              f" · {time.time() - t0:5.0f} s" + ("  ★" if better else ""), flush=True)
        if step == 1_000:
            full = (time.time() - t0) / 1_000 * RECIPE["max_steps"] / 60
            print(f"           (at this speed, the full {RECIPE['max_steps']:,} steps take about {full:.0f} minutes)")
        if bad >= RECIPE["patience"]:
            print(f"early stop: no improvement in {RECIPE['patience']} checks")
            break
    if step == STEPS:
        break
    for group in optimizer.param_groups:                         # this step's learning rate
        group["lr"] = lr_at(step + 1)
    x, y = windows(train_ids, random_starts(train_ids, RECIPE["batch_size"]))   # 64 random windows
    _, loss = model(x, y)                                        # forward: the loss
    optimizer.zero_grad(set_to_none=True)
    loss.backward()                                              # backward: a gradient for every number
    optimizer.step()                                             # update: AdamW nudges every number

seconds = time.time() - t0
model.load_state_dict(best_state)                                # keep the best model
print(f"\ndone in {seconds:.0f} s · best held-back loss {best:.4f} at step {best_step:,}")

## 9 · Watch it learn

Your run is drawn over the course's recorded run (grey), which is read from the repository. On a CPU, your points lie on the
course's line. The dashed line is the course's best check in its whole 100,000-step run.

In [ ]:
from matplotlib.ticker import PercentFormatter

course = json.loads(read(source("tiny-transformer/model/prompter-r1-train.json")))["log"]
course_best = min(e["val"] for e in course)
steps = [e["step"] for e in log]
upto = [e for e in course if e["step"] <= steps[-1]]

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
ax1.plot([e["step"] for e in upto], [e["val"] for e in upto], color="#bbbbbb", lw=6, label="course's run, held back")
ax1.plot(steps, [e["train"] for e in log], "o-", ms=3, label="your run, training")
ax1.plot(steps, [e["val"] for e in log], "o-", ms=3, label="your run, held back")
ax1.axhline(course_best, ls="--", color="grey", lw=1, label=f"course's best check in 100,000 steps ({course_best:.3f})")
ax1.set(title="Loss (lower is better)", xlabel="step", ylabel="cross-entropy", ylim=(1.6, 4.4))
ax1.legend(fontsize=8)
ax2.plot(steps, [e["train_acc"] for e in log], "o-", ms=3, label="training")
ax2.plot(steps, [e["val_acc"] for e in log], "o-", ms=3, label="held back")
ax2.set(title="Accuracy (higher is better)", xlabel="step", ylabel="next characters right")
ax2.yaxis.set_major_formatter(PercentFormatter(1))
ax2.legend(fontsize=8)
every = range(0, RECIPE["max_steps"] + 1, 500)
ax3.plot(list(every), [lr_at(s) for s in every], color="#bbbbbb", label="the whole schedule")
ax3.plot(steps, [e["lr"] for e in log], "o", ms=3, label="your run")
ax3.set(title="Learning rate", xlabel="step")
ax3.legend(fontsize=8)
plt.tight_layout()
plt.show()

recorded = {e["step"]: e["val"] for e in course}
diffs = [abs(e["val"] - recorded[e["step"]]) for e in log if e["step"] in recorded]
print(f"largest difference from the course's held-back loss over {len(diffs)} checks: {max(diffs):.5f}")

## 10 · Evaluate

The checks above used 100 fixed batches. Now the kept model reads the **whole** held-back text once, in back-to-back windows of
32 characters: 111,520 next-character questions, the same measurement the course reports.

- **Accuracy:** how often the most probable character is the right one.
- **Perplexity**, e^loss: the model is as unsure as if it were choosing evenly among that many characters.
- **Bits per character:** the loss divided by ln 2.

In [ ]:
@torch.no_grad()
def whole_text(model, data):
    """Loss and accuracy over a whole text, read in back-to-back windows of T characters (every character scored once)."""
    model.eval()
    total, hits, n = 0.0, 0, 0
    for a in range(0, len(data) - 1, T * 256):                   # 256 windows at a time
        starts = list(range(a, min(a + T * 256, len(data) - T - 1), T))
        if not starts:
            break
        x, y = windows(data, starts)
        logits, loss = model(x, y)
        total += loss.item() * y.numel()
        hits += (logits.argmax(-1) == y).sum().item()
        n += y.numel()
    model.train()
    return total / n, hits / n, n


val_loss, val_acc, n_val = whole_text(model, val_ids)
train_loss, train_acc, _ = whole_text(model, train_ids)
rows = [
    ("an even guess (1 in 65)", math.log(65), 1 / 65),
    ("counting character pairs (the course's F1)", 2.4819, None),
    (f"your model, held back ({n_val:,} questions)", val_loss, val_acc),
    ("your model, training text", train_loss, train_acc),
    ("the course's model after 100,000 steps, held back", 1.9317, 48_930 / 111_520),
]
print(f"{'':52s} {'loss':>6s} {'bits/char':>9s} {'perplexity':>10s} {'accuracy':>8s}")
for name, loss, acc in rows:
    print(f"{name:52s} {loss:6.3f} {loss / math.log(2):9.3f} {math.exp(loss):10.2f} {'' if acc is None else f'{acc:.1%}':>8s}")

## 11 · Make it write

Writing is prediction in a loop:

1. The model gives probabilities for the next character.
2. We draw one character from them and append it.
3. We slide the 32-character window and repeat.

**Temperature** reshapes the probabilities. Below 1, the likely characters get likelier and the text is safer. Above 1, the
choice is more random. The **seed** fixes the random draws, so the same settings always write the same text. Each prompt below is
written twice with the same seed: the same random draws, read through sharper (0.7) or flatter (1.0) probabilities.

In [ ]:
def write(prompt, n=240, temperature=1.0, seed=42):
    """Continue `prompt` by n characters."""
    start = torch.tensor([tok.encode(prompt)], device=device)
    draws = torch.Generator(device=device).manual_seed(seed)
    return tok.decode(model.generate(start, n, temperature=temperature, generator=draws)[0])


samples = []
for prompt, seed in (("ROMEO:\n", 7), ("To be, or not to be", 11)):     # the course's prompts and seeds
    for temperature in (0.7, 1.0):
        samples.append((prompt, temperature, write(prompt, temperature=temperature, seed=seed)))
        print(f"── {prompt!r} · temperature {temperature} ──\n{samples[-1][2]}\n")

In [ ]:
PROMPT = "JULIET:\n"     # your own start: use only characters from the 65 in section 4
print(write(PROMPT, n=300, temperature=0.8, seed=7))

## 12 · Save your model

Everything goes into an `outputs/` folder; in Colab, find it in the Files panel on the left:

| File | What it holds |
|---|---|
| `model.pt` | the trained numbers, the config and the vocabulary: everything needed to rebuild the model |
| `config.json` | sizes, recipe, steps run, best step, time, PyTorch version |
| `train_log.json`, `train_log.csv` | every check: step, learning rate, losses and accuracies |
| `evaluation.json` | section 10's whole-text numbers |
| `loss_curve.png` | section 9's plots |
| `samples.txt` | section 11's writing |

The cell then reloads `model.pt` into a fresh model and checks that it scores the same. In Colab, the next cell zips the folder
and your browser downloads it.

In [ ]:
out = Path("outputs")
out.mkdir(exist_ok=True)
torch.save({"model": model.state_dict(), "config": cfg.to_dict(), "vocab": "".join(tok.chars), "best_step": best_step},
           out / "model.pt")
(out / "config.json").write_text(json.dumps({
    "config": cfg.to_dict(), "params": count_params(model), "recipe": RECIPE, "steps_run": log[-1]["step"],
    "best_step": best_step, "seconds": round(seconds, 1), "device": device, "torch": torch.__version__}, indent=1))
(out / "train_log.json").write_text(json.dumps(log, indent=1))
columns = list(log[0])
(out / "train_log.csv").write_text("\n".join([",".join(columns)] + [",".join(str(e[c]) for c in columns) for e in log]) + "\n")
(out / "evaluation.json").write_text(json.dumps({"held_back": {"loss": val_loss, "accuracy": val_acc, "questions": n_val},
                                                 "training": {"loss": train_loss, "accuracy": train_acc}}, indent=1))
fig.savefig(out / "loss_curve.png", dpi=120)
(out / "samples.txt").write_text("".join(f"--- {p!r} · temperature {t} ---\n{s}\n\n" for p, t, s in samples))

saved = torch.load(out / "model.pt", weights_only=True)      # rebuild the model from the file alone
fresh = Prompter(Config(**saved["config"])).to(device)
fresh.load_state_dict(saved["model"])
assert math.isclose(whole_text(fresh, val_ids)[0], val_loss, rel_tol=1e-6)
print("reloaded outputs/model.pt: same held-back loss ✓\n")
for f in sorted(out.iterdir()):
    print(f"  outputs/{f.name:16s} {f.stat().st_size:>8,} bytes")

In [ ]:
archive = shutil.make_archive("my-tiny-transformer", "zip", out)
print("zipped:", archive)
if "google.colab" in sys.modules:          # in Colab: your browser downloads the zip
    from google.colab import files
    files.download(archive)

## 13 · Try next

- **The full run:** set `QUICK = False` in section 7, then Runtime → Run all. It takes about 16–20 minutes on a CPU, and the
  held-back loss approaches 1.93.
- **A wider model:** in section 5, `cfg = Config(d_model=56, d_ff=112)`. That means more numbers, a lower loss and slower steps. The
  number of heads must divide `d_model`.
- **More heads or blocks:** `Config(n_head=4)` or `Config(n_layer=2)`.
- **A longer context:** `Config(block_size=64)`.
- **Your own text:** read any UTF-8 text file into `text` in section 2 (and drop the fingerprint check). The tokenizer adapts to
  its characters.

Once you change the config or the text, your run no longer follows the course's line; that is the point of the experiment.
Back in the course, T7 shows how to read curves and samples honestly.